# The Duckiedrone IMU data flow

Notebook 4 builds a small program that reads the Duckiedrone's IMU. Before that, it is worth
  knowing where those numbers come from, because they pass through several components before
  they reach that program.

Programs on a robot rarely hand data to each other directly. Instead one program announces its
  data on a named channel, and any program that wants that data asks to be given a copy whenever
  a new data point appears. This system of "named channels" is often referred to as "middleware"
  (because it stands between the hardware, collecting the data, and the software, that consumes
  the data).

A well-known middleware in robotics is the Robot Operating System (ROS). The Duckiedrone uses
  [ROS 2](https://docs.ros.org/), its current version. In ROS 2 the terminology is:
- data channel: **topic**,
- putting data on it: **publishing**,
- asking for copies: **subscribing**,
- a program that takes part in this exchange of data: **node**.

For more information on ROS, check out the
  [Duckiebot ROS LX](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-ros-basics.html).

A ROS 2 node publishes the IMU measurements on a topic, but a topic is not the whole story.
  Behind it sits a chain of hardware, drivers, and filters, and each link can change the data
  before passing it on. This notebook follows that chain on the Duckiedrone, from the chip on
  the flight controller board to the code in a subscriber node. There is no activity in this
  notebook, just the mapping. The chain is the same on the virtual and physical Duckiedrone,
  except that on the virtual Duckiedrone the flight controller and sensors are simulated.

## Intended Learning Outcomes

By the end of the notebook you should be able to answer:

- Which pieces of software touch the IMU data between the sensor chip and a subscriber?

- What does each one add to, or remove from, the signal?

- Where in the chain is the data still raw, and where is it already filtered?

## The IMU on the Duckiedrone

The IMU on the Duckiedrone is an unusual sensor, in that it is embedded in the flight controller
  board, and not stand-alone. This implies that the IMU chip is not accessed directly, but
  through the flight controller. This is actually a desirable feature: PX4 reads the IMU at 8
  kHz, while the ROS 2 topic carries 50 Hz (see the table below).

The flight controller needs these measurements at a high rate to keep the Duckiedrone stable.
  Once the flight controller has used the measurements, it sends them on for nodes to use:

1. The PX4 autopilot software reads the IMU data, applies its configured sensor calibrations,
   and runs its own state estimator (EKF2), which fuses the readings into a Duckiedrone attitude
   estimate.

2. PX4 sends the calibrated IMU measurements and fused attitude over MAVLink.

3. On the companion computer (the Raspberry Pi), the MAVROS2 ROS 2 node receives the MAVLink
   messages and re-publishes them as `sensor_msgs/Imu` on the topic `/mavros/imu/data`. Some
   setups put the Duckiedrone under a namespace, which makes it `/<robot>/mavros/imu/data`
   instead.

Two things follow from this: the orientation that reaches ROS 2 is a fused estimate rather than
  a raw reading, and the calibration work, performed at the time of
  [flight controller configuration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/50-sensor-calibrations.html),
  has already been applied. This notebook walks through this pipeline in detail, to show what
  happens at each step, from the chip to the subscriber.

## The big picture

```
Flight controller board                       | Raspberry Pi
                                              |
              SPI                          MAVLink             ROS 2
+----------+ 8 kHz +---------------------+  50 Hz  +---------+ 50 Hz +-------------+
| IMU chip |------>| PX4                 |----+--->| MAVROS2 |------>| Subscriber  |
+----------+       | - Sensor processing |    |    +---------+       | (Notebook 4)|
                   | - EKF2 estimator    |    |                      +-------------+
                   | - Flight control    |    |
                   +---------------------+    |
                              | 800 Hz        |
                              v               |
                          +--------+          |
                          | Motors |          |
                          +--------+          |
```

The table shows the rates used by the Duckiedrone configuration. Some are controlled by PX4
  parameters, accessible through the
  [QGroundControl](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/42-flight-controller-configuration.html#installing-qgroundcontrol)
  software, while others are part of its internal scheduling or the MAVLink stream setup:

| Stage | Rate | Set by |
|---|---|---|
| IMU chip, sampled by PX4 | 8000 Hz | IMU-driver configuration |
| Rate controller, the loop that flies the Duckiedrone | 800 Hz | PX4 flight-control scheduling, `IMU_GYRO_RATEMAX` |
| Integrated into the form the estimator wants | 250 Hz | `IMU_INTEG_RATE` in this configuration |
| EKF2 estimator update | 100 Hz | EKF2 configuration |
| Streamed over MAVLink to the Raspberry Pi | 50 Hz | MAVLink stream rate |

At a configured stream rate of 50 Hz, the topic publishes one message for every 160 IMU sample
  periods. A physical Duckiedrone should measure about 50 Hz with `ros2 topic hz`. Timing on a
  virtual Duckiedrone can differ, so measure its actual rate with the same command. The last
  section shows where the relevant settings live.

The vertical separator in the diagram marks the two computers. Everything to its left happens on
  the flight controller, a microcontroller running PX4 on a real-time operating system
  ([NuttX](https://github.com/apache/nuttx)), and everything to its right happens on the
  Raspberry Pi running Linux (Ubuntu). On a physical Duckiedrone, a USB connection joins the two
  boards; a virtual Duckiedrone uses a UDP connection instead.

That split is what sets the rates. PX4 uses the IMU in its flight-control loop, which needs
  predictable scheduling on the flight controller. The MAVLink stream is a separate, lower-rate
  feed for the Raspberry Pi. The higher frequency, "low level" loop is in charge of keeping the
  Duckiedrone stable. The higher sensing frequency is necessary due to the high bandwidth of the
  Duckiedrone, i.e., its ability to actuate rapid changes of state. The "outer" control loop,
  governed by the Raspberry Pi, will be responsible for slower changes in state, for example
  navigating from point A to B, therefore not requiring the same high frequency sensing or
  control.

## Step 1. From the IMU chip to PX4

The IMU chip sits on the flight controller board, and PX4 reads it at the high frequency of 8
  kHz (8000 times per second) over the Serial Peripheral Interface (SPI) bus. Nothing on the
  Raspberry Pi is involved in this process: PX4 needs these measurements to stabilize the
  Duckiedrone.

Measurements received by PX4 from the IMU are in the sensor's own frame. The measurements can
  sit slightly off zero even when nothing is moving, and that offset drifts as the board warms
  up. This is the same bias that defeated dead reckoning in Notebook 1. The chip initially
  reports raw counts rather than m/s^2 and rad/s, and its axes follow the board layout.

PX4's sensor-processing pipeline applies the offsets, scale factors, and frame rotation
  configured for the flight controller before downstream components use the data. The resulting
  measurements are expressed in the vehicle frame, FRD (X forward, Y right, Z down). The exact
  calibration values depend on the flight controller and
  [its configuration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/42-flight-controller-configuration.html).

## Step 2. The flight controller's onboard attitude estimator

Notebook 1 introduced fundamental limits of the accelerometer and gyroscope. The gyroscope
  measures angular velocity, but integrating that rate to estimate orientation drifts. The
  accelerometer provides a gravity reference while the Duckiedrone is still or moving gently,
  but it is noisy and cannot distinguish a tilt from a sideways acceleration. PX4 uses both
  measurements in an Extended Kalman Filter called EKF2.

Understanding how a Kalman Filter, or in this case the extended version of it, works is beyond
  the scope of this learning experience. For more, see this other
  [Duckietown learning experience discussing the use of recursive estimation on the Duckiebot, a self-driving car](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-ekf-extended-kalman-filter-localization.html).

Treated as a black box for now, the EKF produces an orientation quaternion (Notebook 2), i.e.,
  the Duckiedrone's estimated orientation with respect to its world frame. PX4 applies the
  values from the
  [sensor calibration](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/50-sensor-calibrations.html),
  and passes calibrated angular rates and accelerations alongside the attitude estimate; those
  are sensor measurements, not outputs of the attitude estimator.

This Duckiedrone hardware configuration has no magnetometer or GPS input, so the estimator has
  no absolute heading measurement. Gravity can correct roll and pitch when the Duckiedrone is
  not accelerating strongly, but yaw is estimated by integrating the gyroscope's angular
  velocity, which is the office-chair problem again: accurate over a turn, slowly wrong over a
  flight. The downward time-of-flight rangefinder can constrain height but does not provide a
  yaw reference either.

## Step 3. From PX4 to MAVROS2 over MAVLink

This is the step where the data crosses the vertical line in the diagram above. PX4 has finished
  with the data and now has to get it to the Raspberry Pi, which on the physical Duckiedrone
  means a USB cable between the two boards and, on the virtual Duckiedrone, a UDP socket
  standing in for it. Either way they speak Micro Air Vehicle Link
  ([MAVLink](https://mavlink.io/en/)), a compact binary protocol built for exactly this kind of
  link, and PX4 streams the fused attitude and the IMU readings across it continuously at 50 Hz.

On the Raspberry Pi, a [MAVROS2](https://github.com/mavlink/mavros) node holds that connection
  open and translates the data, turning each incoming MAVLink message into the equivalent ROS 2
  message. Its `imu` plugin fills in a `sensor_msgs/Imu` and publishes it on:

```
/<robot>/mavros/imu/data
```

The `/<robot>` prefix appears only when the Duckiedrone publishes under a namespace, so on some
  setups the topic is simply `/mavros/imu/data`.

| Field | Where it comes from |
|---|---|
| `orientation` (quaternion) | the fused attitude estimate from EKF2 |
| `angular_velocity` | gyroscope, bias-corrected and frame-converted |
| `linear_acceleration` | accelerometer, bias-corrected and frame-converted |
| `*_covariance` | uncertainty when provided; all zeros mean unknown, and `orientation_covariance[0] = -1` means no orientation estimate |

MAVROS2 does one more important manipulation on this data: it adjusts the reference frame
  convention. PX4 works in FRD, but ROS works in FLU (X forward, Y left, Z up), so MAVROS2 flips
  the Y and Z axes as it converts the data from PX4. That flip leaves X alone: roll rate and
  forward acceleration keep the sign PX4 gave them, while pitch rate, yaw rate and the Y and Z
  accelerations all come out with inverted signs.

The orientation quaternion crosses a second boundary on top of that one, from the NED world
  frame (X north, Y east, Z down) PX4 works in to the ENU frame (East-North-Up) ROS expects.

Note also what the first row of the table implies. `angular_velocity` and `linear_acceleration`
  are measurements that have been cleaned up, but `orientation` is the output of an algorithm
  (EKF2), so it carries the drift from Step 2 with it. To inspect a message without the fused
  orientation, use `/mavros/imu/data_raw`. Its acceleration and gyroscope fields come from
  `HIGHRES_IMU`, so they are not direct chip readings: PX4 has already applied the calibrations
  to them and may subtract EKF2's current bias estimate before MAVROS2 frame-converts them.

## Step 4. From MAVROS2 to the subscriber node

The subscriber node is the last stop. It subscribes to the ROS 2 `/<robot>/mavros/imu/data`
  topic and hands over a function to go with it. From then on, every time MAVROS2 publishes a
  new message, ROS 2 calls that function and passes it the message. A function that gets called
  when a message arrives is a **callback**, and filling one in is a central part of Notebook 4.

The node receives a `sensor_msgs/Imu` whose accelerations and angular rates have already been
  calibrated, scaled, and frame-converted. Its orientation is the estimator's fused output. The
  node does not need to repeat raw-sensor calibration; its callback unpacks the values, converts
  the orientation quaternion to angles, and stores the result for display.

The caveat is the one from Step 2: the orientation is the flight controller's best estimate, so
  it includes the estimator's limitations as well as its filtering. The MAVLink stream rate sets
  how often it arrives, and that is much lower than the IMU sampling rate. Notebook 4 measures
  the actual topic rate with `ros2 topic hz` rather than assuming a nominal value.

## Why not a DTPS driver, like the other sensors?

Other sensors on the Duckiedrone, such as the time-of-flight rangefinder and camera, reach the
  agent code, the code that controls the Duckiedrone, through a DTPS
  ([Duckietown Postal Service](https://docs.duckietown.com/ente/duckietown-manual/04-software-tools/duckietown-postal-service-dtps.html))
  driver and a ROS 2 bridge. The IMU skips all of that and arrives straight from MAVROS2. There
  are three closely related reasons:

1. The flight controller already owns the chip. It is soldered to the flight controller board
   and read by PX4 firmware, so no driver on the Raspberry Pi touches it. A DTPS driver would
   ask PX4 for the data and republish it, which overlaps with MAVROS2's role.

2. The fast loop has to stay on the microcontroller. PX4 corrects the Duckiedrone's attitude
   every few milliseconds, and a general-purpose operating system cannot guarantee the same
   scheduling. Routing the IMU to the Raspberry Pi to close that loop would add latency and
   uncertainty.

3. The attitude estimate is already fused before it crosses over. PX4 completes the estimation
   before the Raspberry Pi receives it, so MAVROS2 avoids duplicating the estimator and gives
   ROS 2 consumers one shared attitude estimate.

On the Duckiedrone, then, the flight controller is the IMU driver and MAVROS2 is the ROS 2
  bridge. That is useful for troubleshooting: the chip and PX4 continue operating whether or not
  anything is subscribed, so a quiet `/<robot>/mavros/imu/data` topic points to MAVROS2 or the
  link to PX4 rather than directly to the sensor.

## Changing any of this

Several rates in the table are configurable, while others are part of PX4's internal scheduling.
  Suppose an application needs an IMU topic faster than 50 Hz for finer detail. Which number
  should change?

`IMU_INTEG_RATE` controls one upstream rate. Changing it alone does not change either ROS 2
  topic rate, because those rates are set at the MAVLink boundary. The diagram shows why these
  settings affect different parts of the pipeline.

### Settings the flight controller remembers

`IMU_INTEG_RATE` and `IMU_GYRO_RATEMAX` are PX4 parameters. A parameter is written into the
  flight controller's persistent storage, so projects commonly load their desired values from a
  parameter file. The values in the table are those used by the Duckiedrone configuration; PX4
  defaults can vary by version, board, and airframe.

They can be read and changed in QGroundControl under *Vehicle Setup > Parameters*, either one at
  a time or by loading a whole `.params` file. Check the parameter description for the current
  PX4 release to learn whether a reboot is required. The
  [PX4 parameter reference](https://docs.px4.io/main/en/advanced_config/parameter_reference.html)
  lists available parameters, and
  [Finding/Updating Parameters](https://docs.px4.io/main/en/advanced_config/parameters.html)
  covers how to find and change them.

These parameters operate on the flight-controller side of the diagram. They can change the
  source data PX4 prepares, but they do not directly set the MAVLink publication rate.

### What actually crosses the cable

Sending is a separate decision. When the link to the Raspberry Pi comes up, PX4 configures which
  MAVLink messages it will send and how often to send each one. The IMU uses two of them:
  `HIGHRES_IMU` carries the accelerometer and gyroscope, and `ATTITUDE_QUATERNION` carries the
  fused orientation and angular rate.

Those messages feed different ROS 2 topics. MAVROS2 publishes `/mavros/imu/data_raw` when
  `HIGHRES_IMU` arrives. It publishes `/mavros/imu/data` when `ATTITUDE_QUATERNION` arrives,
  combining its orientation and angular rate with the latest acceleration from `HIGHRES_IMU`. If
  the two messages run at different rates, a `data` message can therefore contain a new attitude
  and a previously received acceleration.

Each MAVLink message interval directly controls its corresponding ROS 2 output: `data_raw`
  follows `HIGHRES_IMU`, while `data` follows `ATTITUDE_QUATERNION`. To change a particular
  topic rate, adjust its source-message interval rather than an unrelated upstream parameter.
  [MAVLink Messaging](https://docs.px4.io/main/en/middleware/mavlink.html) describes how PX4
  configures these streams.

### Asking for a different rate while everything is running

A reboot is not needed to request a different stream rate. The Raspberry Pi can ask the flight
  controller for one while it is running, using the MAVLink `SET_MESSAGE_INTERVAL` command
  (number 511). The command takes the message id and the interval in microseconds. MAVROS2
  exposes that command as a ROS 2 service, so a terminal is enough.

If the Duckiedrone publishes under a namespace, replace `/mavros` in the service and topic paths
  below with `/<robot>/mavros`.

`HIGHRES_IMU` is message id 105, and 100000 microseconds is a tenth of a second, so this
  requests 10 Hz for `/mavros/imu/data_raw`:

```bash
ros2 service call /mavros/cmd/command mavros_msgs/srv/CommandLong \
  "{command: 511, param1: 105.0, param2: 100000.0}"
```

Leave `ros2 topic hz /mavros/imu/data_raw` running in another terminal while sending it. The
  measured rate should approach 10 Hz; timing on a virtual Duckiedrone can introduce variation.
  Send `20000.0` to request 50 Hz again. This request does not set `/mavros/imu/data` to 10 Hz:
  that topic is published by `ATTITUDE_QUATERNION`, whose interval must be requested separately
  when an application needs both topics at the same rate.

Unlike a parameter, this request is not persistent. Reboot the flight controller and the rate
  returns to the link's configured value. The MAVROS2 launcher uses the same command for message
  id 132, `DISTANCE_SENSOR`, with an interval of `-1` to disable that stream when it is not
  needed. The request must be sent again on every launch. The message ids, and the meaning of
  each command parameter, are in the
  [MAVLink common message set](https://mavlink.io/en/messages/common.html).

### One switch that looks like a rate and is not

One more setting sits on the Raspberry Pi. MAVROS2 loads a list of plugins from
  `px4_config.yaml`, and a plugin turns one family of MAVLink messages into ROS 2 topics. The
  `imu` plugin creates `/mavros/imu/data` at root, or `/<robot>/mavros/imu/data` when MAVROS2
  uses that namespace.

Removing that plugin removes its ROS 2 topics, but PX4 continues sending the corresponding
  MAVLink messages at the configured rate. Disabling the plugin stops ROS 2 from consuming the
  messages; it does not reduce traffic on the link. To reduce that traffic, change the
  corresponding MAVLink message interval.